In [ ]:
import datetime
import os

import gymnasium as gym
import numpy as np
import torch
from tianshou.algorithm import PPO
from tianshou.algorithm.algorithm_base import Algorithm
from tianshou.algorithm.modelfree.reinforce import ProbabilisticActorPolicy
from tianshou.algorithm.optim import AdamOptimizerFactory, LRSchedulerFactoryLinear
from tianshou.data import Collector, CollectStats, VectorReplayBuffer
from tianshou.env import DummyVectorEnv, VectorEnvNormObs
from tianshou.trainer import OnPolicyTrainerParams
from tianshou.utils import TensorboardLogger
from tianshou.utils.net.common import ActorCritic, Net
from tianshou.utils.net.continuous import ContinuousActorProbabilistic, ContinuousCritic
from tianshou.utils.space_info import SpaceInfo
from torch import nn
from torch.distributions import Distribution, Independent, Normal
from torch.utils.tensorboard import SummaryWriter


## Hyperparms

In [2]:
ENV_NAME = "LunarLander-v3"
ENV_NUM = 8
LR = 1.5e-3

EPOCH = 1000
EPOCH_NUM_STEP = 1024
COLLECTION_STEP_NUM_ENV_STEPS=2048
K_EPOCH = 4

GAMMA = .99
GAE_LAMBDA = .9
EPS_CLIP = .2
MAX_GRAD_NORM = .5
VF_COEF = .5
ENTROPY_COEF = 0

BUFFER_SIZE: int = 4096
BATCH_SIZE: int = 64

LOG_DIR="./logs"
CHECKPOINTS_DIR = "./data/checkpoints/"

## Envs init

In [4]:
def make_env():
    return gym.make(ENV_NAME, continuous=True)

In [5]:
env = make_env()
space_info = SpaceInfo.from_env(env)

state_shape = space_info.observation_info.obs_shape
action_shape = space_info.action_info.action_shape
max_action = space_info.action_info.max_action

space_info, state_shape

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


(SpaceInfo(action_info=ActionSpaceInfo(action_shape=(2,), min_action=-1.0, max_action=1.0), observation_info=ObservationSpaceInfo(obs_shape=(8,))),
 (8,))

In [6]:
training_envs = VectorEnvNormObs(DummyVectorEnv(
    [make_env for _ in range(ENV_NUM)]
))

test_envs = VectorEnvNormObs(
    DummyVectorEnv([make_env for _ in range(ENV_NUM)]),
    update_obs_rms=False,
)

test_envs.set_obs_rms(training_envs.get_obs_rms())


## PPO init

In [7]:
hidden_sizes = [64, 64]
device = "cpu"
# device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
device

'cpu'

In [8]:
net_a = Net(
    state_shape=state_shape,
    hidden_sizes=hidden_sizes,
    activation=nn.Tanh,
)
net_c = Net(
    state_shape=state_shape,
    hidden_sizes=hidden_sizes,
    activation=nn.Tanh,
)

In [9]:
actor = ContinuousActorProbabilistic(
    preprocess_net=net_a,
    action_shape=action_shape,
    unbounded=True,
).to(device)

In [10]:
critic = ContinuousCritic(preprocess_net=net_c).to(device)

In [11]:
actor_critic = ActorCritic(actor, critic)

In [12]:
import math

math.exp(-0.5)

0.6065306597126334

### Orthogonal init

In [13]:
torch.nn.init.constant_(actor.sigma_param, -0.5)
for m in actor_critic.modules():
    if isinstance(m, torch.nn.Linear):
        # orthogonal initialization
        torch.nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
        torch.nn.init.zeros_(m.bias)

In [14]:
for m in actor.mu.modules():
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.zeros_(m.bias)
        m.weight.data.copy_(0.01 * m.weight.data)

## Optimizer init

In [15]:
optim = AdamOptimizerFactory(lr=LR)
optim.with_lr_scheduler_factory(
    LRSchedulerFactoryLinear(
        max_epochs=EPOCH,
        epoch_num_steps=EPOCH_NUM_STEP,
        collection_step_num_env_steps=COLLECTION_STEP_NUM_ENV_STEPS,
    )
)

AdamOptimizerFactory[id=5615510464, lr_scheduler_factory=LRSchedulerFactoryLinear[num_epochs=1000, epoch_num_steps=1024, collection_step_num_env_steps=2048], lr=0.0015, weight_decay=0, eps=1e-08, betas=(0.9, 0.999)]

## Distribution

In [16]:
def dist(loc_scale: tuple[torch.Tensor, torch.Tensor]) -> Distribution:
    loc, scale = loc_scale
    return Independent(Normal(loc, scale), 1)

## Policy

In [ ]:
policy = ProbabilisticActorPolicy(
    actor=actor,
    dist_fn=dist,
    action_scaling=True,
    action_bound_method="clip", # torch.clamp(-1, 1)
    action_space=env.action_space,
)

In [18]:
algorithm: PPO = PPO(
    policy=policy,
    critic=critic,
    optim=optim,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    max_grad_norm=MAX_GRAD_NORM,
    vf_coef=VF_COEF,
    ent_coef=ENTROPY_COEF,
    return_scaling=False,
    eps_clip=EPS_CLIP,
    value_clip=True,
    dual_clip=None,
    advantage_normalization=True,
    recompute_advantage=True,
)

In [19]:
buffer = VectorReplayBuffer(BUFFER_SIZE, len(training_envs))

## Collector
[doc](https://tianshou.org/en/stable/01_user_guide/02_core_abstractions.html#collector)

In [20]:
training_collector = Collector[CollectStats](
    algorithm, training_envs, buffer, exploration_noise=True
)
test_collector = Collector[CollectStats](algorithm, test_envs)

### Log

In [21]:
now = datetime.datetime.now().strftime("%y%m%d-%H%M%S")
log_path = os.path.join(LOG_DIR, ENV_NAME, f"ppo_tianshou_{now}")
writer = SummaryWriter(log_path)
logger = TensorboardLogger(writer)

In [22]:
def save_best_fn(policy: Algorithm) -> None:
    state = {"model": policy.state_dict(), "obs_rms": training_envs.get_obs_rms()}
    torch.save(state, os.path.join(CHECKPOINTS_DIR, f"policy_{ENV_NAME}.pth"))

# Training

In [23]:
%%capture
result = algorithm.run_training(
    OnPolicyTrainerParams(
        training_collector=training_collector,
        test_collector=test_collector,
        max_epochs=EPOCH,
        epoch_num_steps=EPOCH_NUM_STEP,
        update_step_num_repetitions=K_EPOCH,
        test_step_num_episodes=ENV_NUM,
        batch_size=BATCH_SIZE,
        collection_step_num_env_steps=COLLECTION_STEP_NUM_ENV_STEPS,
        save_best_fn=save_best_fn,
        logger=logger,
        test_in_training=False,
    )
)


In [31]:
test_collector.reset()
s = test_collector.collect(n_episode=ENV_NUM, render=False)
print(f"reward: {s.returns_stat.mean:.1f} ± {s.returns_stat.std:.1f}")
print(f"length: {s.lens_stat.mean:.1f}")
print(f"episodes: {s.n_collected_episodes}")

reward: 286.2 ± 17.1
length: 182.1
episodes: 8


In [26]:
class ONNXActor(torch.nn.Module):
    def __init__(self, actor, obs_rms, max_action=1.0):
        super().__init__()
        self.actor = actor
        self.register_buffer("mean", torch.as_tensor(obs_rms.mean, dtype=torch.float32))
        self.register_buffer("var", torch.as_tensor(obs_rms.var, dtype=torch.float32))
        self.eps = float(obs_rms.eps)
        self.clip_max = float(obs_rms.clip_max)
        self.max_action = max_action

    def forward(self, obs):
        obs = (obs - self.mean) / torch.sqrt(self.var + self.eps)
        obs = torch.clamp(obs, -self.clip_max, self.clip_max)
        (mu, _), _ = self.actor(obs)
        return torch.tanh(mu) * self.max_action

In [28]:
model = ONNXActor(policy.actor, training_envs.get_obs_rms()).eval()

In [32]:
%%capture

out_name = "action"

torch.onnx.export(
    model,
    torch.randn(1, state_shape[0]),
    "./data/latest.onnx",
    input_names=["obs"], output_names=[out_name],
    dynamic_axes={"obs": {0: "batch"}, out_name: {0: "batch"}},
    external_data=False,
)